# National LMIS — 03: Demand-Supply Index & Skill Gap Engine
**Ministry of Skill Development & Entrepreneurship (MSDE) | SIH_26246**

This notebook computes the standardized Demand Index, Supply Index, and Skill Gap classification:
- **Demand Index (0-100)**: 60% Job Postings Volume + 40% Hiring Growth Rate
- **Supply Index (0-100)**: 50% Certified Trainees + 20% Capacity Utilization + 30% Placement Rate
- **Net Skill Gap**: `Demand - Supply`
- **Status Classification**: `SHORTAGE` (> +15%), `BALANCED` (±15%), `OVERSUPPLY` (< -15%)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'


### 1. Load Clean Datasets


In [ ]:
df_jobs = pd.read_csv(PROCESSED_DIR / 'cleaned_job_postings.csv')
df_train = pd.read_csv(PROCESSED_DIR / 'cleaned_training_data.csv')
df_place = pd.read_csv(PROCESSED_DIR / 'cleaned_placements.csv')
# Aggregate latest demand snapshot
latest_month = df_jobs['month'].max()
df_demand_snap = df_jobs[df_jobs['month'] == latest_month].groupby(['state', 'district', 'sector', 'trade'], as_index=False).agg({
    'job_postings': 'sum',
    'hiring_growth': 'mean'
})
print(f'Demand snapshot for {latest_month}: {len(df_demand_snap)} records.')


### 2. Compute Safe Min-Max Normalization


In [ ]:
def safe_min_max(series):
    min_val, max_val = series.min(), series.max()
    if max_val == min_val:
        return pd.Series(0.5, index=series.index)
    return (series - min_val) / (max_val - min_val)

df_demand_snap['norm_postings'] = safe_min_max(df_demand_snap['job_postings'])
df_demand_snap['norm_growth'] = safe_min_max(df_demand_snap['hiring_growth'])
df_demand_snap['demand_index'] = (0.60 * df_demand_snap['norm_postings'] + 0.40 * df_demand_snap['norm_growth']) * 100
print('Demand Index Summary:')
print(df_demand_snap['demand_index'].describe())


### 3. Compute Supply Index


In [ ]:
df_supply_merged = pd.merge(df_train, df_place, on=['state', 'district', 'sector', 'trade'], how='inner')
df_supply_merged['norm_certified'] = safe_min_max(df_supply_merged['candidates_certified'])
df_supply_merged['norm_seats'] = safe_min_max(df_supply_merged['training_seats'])
df_supply_merged['norm_placement'] = safe_min_max(df_supply_merged['placement_rate'])
df_supply_merged['supply_index'] = (0.50 * df_supply_merged['norm_certified'] + 0.20 * df_supply_merged['norm_seats'] + 0.30 * df_supply_merged['norm_placement']) * 100
print('Supply Index Summary:')
print(df_supply_merged['supply_index'].describe())


### 4. Merge Demand & Supply to Form Skill Gap Engine


In [ ]:
merged = pd.merge(df_demand_snap, df_supply_merged, on=['state', 'district', 'sector', 'trade'], how='inner')
merged['demand'] = merged['job_postings']
merged['supply'] = merged['candidates_certified']
merged['gap'] = merged['demand'] - merged['supply']
merged['gap_percentage'] = ((merged['gap'] / merged['demand'].replace(0, np.nan)) * 100).fillna(0.0).round(1)

def classify_status(gap_pct):
    if gap_pct > 15.0:
        return 'SHORTAGE'
    elif gap_pct < -15.0:
        return 'OVERSUPPLY'
    return 'BALANCED'

merged['gap_status'] = merged['gap_percentage'].apply(classify_status)
print('Gap Status Distribution:')
print(merged['gap_status'].value_counts())
display(merged[['state', 'district', 'trade', 'demand', 'supply', 'gap', 'gap_percentage', 'gap_status']].head(10))
